# 05 — Terraform Plan-Diff Safety Gate

Companion notebook to `07-production-resilience-and-operational-engineering.md`. Chapter 07's
hardening-gap section names this notebook directly: *"Notebook `05_terraform_plan_diff_safety_gate.ipynb`
implements a version of exactly this check, standalone."*

The gap this notebook closes: chapter 07 names, candidly, that **"a `terraform plan` showing an
unexpected `destroy` on a stateful resource must be manually reviewed before approval is enforced by
reviewer diligence and team convention, not by an automated policy gate."** Nothing in the pipeline as
described actually inspects the plan's JSON output and blocks approval if it contains a
`delete`/replace targeting a resource type known to hold live data.

This notebook builds that automated policy check: a function that parses a simplified
`terraform show -json`-style plan (reusing the `resource_changes` shape from
`02_terraform_basics_demo.ipynb`), classifies each destroyed/replaced resource by whether its
resource type is "stateful," and fails the check outright -- no silent pass, no human-diligence
dependency -- if an unreviewed destroy targets one.

Fully offline: standard library only (no real `terraform` CLI, no real Azure subscription, no network
calls -- exactly the same constraint `02_terraform_basics_demo.ipynb` states for the same reason).

In [1]:
import json

print("Imports OK")

Imports OK


## Step 1 — Which resource types are "stateful" (chapter 07's hardening-gap list)

Chapter 07 names three examples directly: a Key Vault, an Azure SQL database, a storage account. This
notebook keeps that as an explicit, editable allow-list rather than a heuristic -- the same "deny by
default for anything not explicitly reviewed" instinct chapter 07 (and course 04's chapter 07, for a
different resource) argues for elsewhere in this curriculum: a new resource type is treated as
stateful-until-proven-otherwise is *not* what's implemented here (this list is explicit, not a
default-safe fallback), which is itself worth naming as a limitation in Step 5 below.

In [2]:
STATEFUL_RESOURCE_TYPE_PREFIXES = (
    "azurerm_key_vault",
    "azurerm_mssql_database",
    "azurerm_sql_database",
    "azurerm_storage_account",
    "azurerm_cosmosdb_account",
)


def is_stateful_resource(address: str) -> bool:
    resource_type = address.split(".", 1)[0]
    return resource_type.startswith(STATEFUL_RESOURCE_TYPE_PREFIXES)


# Sanity checks against the exact examples chapter 07 names.
assert is_stateful_resource("azurerm_key_vault.kv")
assert is_stateful_resource("azurerm_mssql_database.appdb")
assert is_stateful_resource("azurerm_storage_account.docs")
assert not is_stateful_resource("azurerm_linux_web_app.app")
assert not is_stateful_resource("azurerm_service_plan.plan")
print("Stateful-resource classifier defined and sanity-checked")

Stateful-resource classifier defined and sanity-checked


## Step 2 — The policy check itself

Reuses `02_terraform_basics_demo.ipynb`'s simplified plan shape: a list of `resource_changes`, each
with a `change.actions` list. A destroy is either `["delete"]` outright, or `["create", "delete"]` /
`["delete", "create"]` for a replace (Terraform tears down and recreates in one step -- still a real
destroy of the existing resource, and still exactly the case chapter 07 warns about, since a replace
on a stateful resource means the *existing* data-holding instance is destroyed even though a new one
immediately takes its place).

In [3]:
def destroys_resource(change_actions: list) -> bool:
    return "delete" in change_actions


def check_plan_for_unreviewed_stateful_destroys(plan: dict) -> dict:
    """The automated policy check chapter 07 says is missing. Returns a dict with 'blocked'
    (bool) and 'flagged_resources' (list of {address, actions, resource_type})."""
    flagged = []
    for rc in plan.get("resource_changes", []):
        actions = rc.get("change", {}).get("actions", [])
        if destroys_resource(actions) and is_stateful_resource(rc["address"]):
            flagged.append({
                "address": rc["address"],
                "actions": actions,
                "resource_type": rc["address"].split(".", 1)[0],
            })
    return {"blocked": len(flagged) > 0, "flagged_resources": flagged}


print("Policy check defined")

Policy check defined


## Step 3 — A safe plan: only stateless resources change

The healthy case -- an App Service update and a new Service Plan, nothing stateful touched. The gate
must not block a routine, harmless change.

In [4]:
safe_plan = {
    "resource_changes": [
        {"address": "azurerm_service_plan.plan", "change": {"actions": ["update"]}},
        {"address": "azurerm_linux_web_app.app", "change": {"actions": ["update"]}},
        {"address": "azurerm_linux_web_app.app.app_settings", "change": {"actions": ["update"]}},
        {"address": "azurerm_service_plan.old_plan", "change": {"actions": ["delete"]}},  # stateless -- fine to destroy
    ]
}

result = check_plan_for_unreviewed_stateful_destroys(safe_plan)
print(result)
assert result["blocked"] is False, "a plan with no stateful destroys must not be blocked"
print("\nConfirmed: the safe plan passes -- destroying a stale App Service Plan is routine cleanup, "
      "not a data-loss risk.")

{'blocked': False, 'flagged_resources': []}

Confirmed: the safe plan passes -- destroying a stale App Service Plan is routine cleanup, not a data-loss risk.


## Step 4 — The exact scenario chapter 07 warns about: an unexpected Key Vault destroy

A plan that -- plausibly, per the chapter -- results from someone renaming a `azurerm_key_vault`
resource block without a `moved` block (Terraform sees it as "destroy the old address, create a new
one" rather than a rename), or a module refactor that inadvertently changes a Key Vault's implicit
resource address. Either way: a live secret store is about to be destroyed, and under today's
process this is visible in the plan output but not *blocked* by anything except a reviewer noticing.

In [5]:
dangerous_plan = {
    "resource_changes": [
        {"address": "azurerm_resource_group.rg", "change": {"actions": ["no-op"]}},
        {"address": "azurerm_linux_web_app.app", "change": {"actions": ["update"]}},
        # A Key Vault rename that Terraform sees as destroy-and-recreate, not a rename -- the
        # single most dangerous line in this plan, easy to miss scrolling past dozens of no-ops.
        {"address": "azurerm_key_vault.kv", "change": {"actions": ["delete", "create"]}},
        {"address": "azurerm_key_vault.kv_renamed", "change": {"actions": ["create"]}},
        {"address": "azurerm_mssql_database.appdb", "change": {"actions": ["no-op"]}},
    ]
}

result = check_plan_for_unreviewed_stateful_destroys(dangerous_plan)
print(json.dumps(result, indent=2))

assert result["blocked"] is True, "a plan destroying a live Key Vault must be blocked outright"
assert result["flagged_resources"][0]["address"] == "azurerm_key_vault.kv"
print("\nConfirmed: the gate blocks this plan outright -- turning 'the reviewer is supposed to "
      "catch this' into 'the pipeline cannot proceed past this point regardless of who's reviewing "
      "or how busy they are,' exactly chapter 07's proposed fix.")

{
  "blocked": true,
  "flagged_resources": [
    {
      "address": "azurerm_key_vault.kv",
      "actions": [
        "delete",
        "create"
      ],
      "resource_type": "azurerm_key_vault"
    }
  ]
}

Confirmed: the gate blocks this plan outright -- turning 'the reviewer is supposed to catch this' into 'the pipeline cannot proceed past this point regardless of who's reviewing or how busy they are,' exactly chapter 07's proposed fix.


## Step 5 — Wiring this into the pipeline: a required, blocking stage before the approval gate

Chapter 07: this check should run as a **required, blocking pipeline stage before the approval gate**
-- failing the stage outright, with no human override short of an explicit, logged exception process.
`run_policy_gate()` below is that stage, in miniature: it either lets the plan proceed to the human
approval step, or fails the stage and requires the same kind of explicit, logged override this
course's other notebooks use for their own "block by default, override with a reason" gates (see
`04_version_compatibility_matrix_validator.ipynb` in course 06's own notebooks folder for the same
pattern applied to a different check).

In [6]:
def run_policy_gate(plan: dict, override: bool = False, override_reason: str = None) -> dict:
    check = check_plan_for_unreviewed_stateful_destroys(plan)
    if not check["blocked"]:
        return {"proceeds_to_approval_gate": True, "overridden": False, "flagged_resources": []}

    if override:
        if not override_reason:
            raise ValueError("an override MUST be accompanied by a logged reason -- no silent overrides")
        print(f"WARNING: proceeding past a stateful-destroy gate under explicit, logged override: {override_reason}")
        return {"proceeds_to_approval_gate": True, "overridden": True, "flagged_resources": check["flagged_resources"]}

    return {"proceeds_to_approval_gate": False, "overridden": False, "flagged_resources": check["flagged_resources"]}


safe_outcome = run_policy_gate(safe_plan)
print("Safe plan:", safe_outcome)
assert safe_outcome["proceeds_to_approval_gate"] is True

blocked_outcome = run_policy_gate(dangerous_plan)
print("Dangerous plan, no override:", blocked_outcome)
assert blocked_outcome["proceeds_to_approval_gate"] is False

overridden_outcome = run_policy_gate(
    dangerous_plan, override=True,
    override_reason="Confirmed with the Key Vault owner this is a planned rotation with secrets pre-migrated; "
                     "logged as change ticket CHG-4821.",
)
print()
print("Dangerous plan, explicit override:", overridden_outcome)
assert overridden_outcome["proceeds_to_approval_gate"] is True and overridden_outcome["overridden"] is True
print("\nConfirmed: the only way past a flagged stateful destroy is an explicit, logged exception -- "
      "never an unattended pass.")

Safe plan: {'proceeds_to_approval_gate': True, 'overridden': False, 'flagged_resources': []}
Dangerous plan, no override: {'proceeds_to_approval_gate': False, 'overridden': False, 'flagged_resources': [{'address': 'azurerm_key_vault.kv', 'actions': ['delete', 'create'], 'resource_type': 'azurerm_key_vault'}]}

Dangerous plan, explicit override: {'proceeds_to_approval_gate': True, 'overridden': True, 'flagged_resources': [{'address': 'azurerm_key_vault.kv', 'actions': ['delete', 'create'], 'resource_type': 'azurerm_key_vault'}]}

Confirmed: the only way past a flagged stateful destroy is an explicit, logged exception -- never an unattended pass.


## Tying it back

- This is a small, well-scoped check -- exactly chapter 07's own framing: *"this wasn't automated
  because building a policy engine felt like more upfront investment than the number of actual close
  calls justified at the time... but the fix is a single, well-scoped pipeline stage."* Steps 1-2 are
  that stage.
- The honest limitation worth naming, the same way chapter 07 names its own: `STATEFUL_RESOURCE_TYPE_PREFIXES`
  is an explicit allow-list, not a default-safe classifier -- a new stateful resource type introduced
  by a future Azure provider update (or a custom/community provider) that isn't added to this list
  would silently **not** be flagged, which is the same "accessible/unprotected until someone remembers
  to add a rule" failure shape as course 04's chapter 07 RLS bug narrative (a new table/resource is
  unprotected by default, rather than protected by default). A more defensive version would invert
  this: treat every resource type as stateful unless it's on an explicit *known-stateless* allow-list
  (`azurerm_service_plan`, `azurerm_role_assignment`, and similar), so an unrecognized future resource
  type fails safe (gets flagged for review) rather than fails open (passes silently).
- Step 5's override path is deliberately the same shape as `04_version_compatibility_matrix_validator.ipynb`'s
  override path in this same course -- one consistent "block by default, only proceed on an explicit,
  logged exception" pattern used for two structurally similar problems (an unsafe version combination,
  an unsafe infrastructure change), rather than two different ad hoc mechanisms.